In [1]:
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 47.5 MB/s eta 0:00:00


In [2]:
import os, yaml, shutil
from ultralytics import YOLO

# ── Configuration ──────────────────────────────────────────────
DATASET_PATH = '/kaggle/input/your-helmet-dataset'   # <-- CHANGE THIS
PROJECT_NAME = 'helmet_detection'
MODEL_BASE   = 'yolov8m.pt'
EPOCHS       = 80
IMG_SIZE     = 640
BATCH_SIZE   = 16
# ───────────────────────────────────────────────────────────────

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
# Create data.yaml file

yaml_content = """
path: /kaggle/input/datasets/abhishek162kumar/helmet-detection-3  # change dataset name

train: train/images
val: valid/images
test: test/images

nc: 2
names: ['With Helmet', 'Without Helmet']
"""

# Save YAML file in working directory
with open('/kaggle/working/data.yaml', 'w') as f:
    f.write(yaml_content)

print("✅ data.yaml created at /kaggle/working/data.yaml")

✅ data.yaml created at /kaggle/working/data.yaml


In [4]:
data_yaml = '/kaggle/working/data.yaml'

model = YOLO(MODEL_BASE)

results = model.train(
    data    = data_yaml,
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH_SIZE,
    project = '/kaggle/working/runs',
    name    = PROJECT_NAME,
    device  = 0,
    patience= 20,
    save    = True,
    plots   = True,
    # Helmet detection: head is small — increase scale sensitivity
    scale   = 0.5,
    translate = 0.1,
    degrees = 10.0,
    fliplr  = 0.5,
    mosaic  = 0.7,
    copy_paste = 0.2,
    close_mosaic = 10,
)

Ultralytics 8.4.38 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=0.7, multi_scale=0.0, name=helmet_detection, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience

In [9]:
best_weights = f'/kaggle/working/runs/{PROJECT_NAME}/weights/best.pt'
model_best = YOLO(best_weights)
metrics = model_best.val(data=data_yaml, imgsz=IMG_SIZE)
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)

Ultralytics 8.4.38 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,840,918 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 106.6±61.0 MB/s, size: 64.5 KB)
val: Scanning /kaggle/input/datasets/abhishek162kumar/helmet-detection-3/valid/labels... 421 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 421/421 807.8it/s 0.5s0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/abhishek162kumar/helmet-detection-3/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 2.2it/s 12.3s0.4s
                   all        421        702      0.881      0.854      0.929       0.58
           With Helmet        237        356      0.892      0.907      0.955      0.638
        Without Helmet        232        346      0.871      0.801      0.903      0.522
Speed: 1.4ms preprocess, 23.7ms inference, 0.0ms lo

In [10]:
output_path = '/kaggle/working/model3_helmet.pt'
shutil.copy(best_weights, output_path)
print(f'Model saved: {output_path}')

Model saved: /kaggle/working/model3_helmet.pt
